> **MIT License** - Copyright (c) 2026 QuantWall. Full text: https://opensource.org/license/mit
>
> **Disclaimer.** This content is general information and software-engineering education only. It is not financial product advice, a recommendation, or an offer to buy or sell any financial product, and it does not take into account your objectives, financial situation or needs. The author does not hold an Australian Financial Services Licence and is not a registered Commodity Trading Advisor. Trading futures and leveraged products carries a high risk of loss. Consult a licensed adviser before making any financial decision.
>
> **HYPOTHETICAL OR SIMULATED PERFORMANCE RESULTS HAVE CERTAIN LIMITATIONS. UNLIKE AN ACTUAL PERFORMANCE RECORD, SIMULATED RESULTS DO NOT REPRESENT ACTUAL TRADING. ALSO, SINCE THE TRADES HAVE NOT BEEN EXECUTED, THE RESULTS MAY HAVE UNDER-OR-OVER COMPENSATED FOR THE IMPACT, IF ANY, OF CERTAIN MARKET FACTORS, SUCH AS LACK OF LIQUIDITY. SIMULATED TRADING PROGRAMS IN GENERAL ARE ALSO SUBJECT TO THE FACT THAT THEY ARE DESIGNED WITH THE BENEFIT OF HINDSIGHT. NO REPRESENTATION IS BEING MADE THAT ANY ACCOUNT WILL OR IS LIKELY TO ACHIEVE PROFIT OR LOSSES SIMILAR TO THOSE SHOWN.**
>
> All data shown is synthetic or publicly available sample data. Features are standard textbook baselines (lagged returns, ATR, RSI, moving averages, volume). No proprietary signal is disclosed or implied.

# What a real edge looks like: every number in the video

Each cell reproduces one claim from the video, from a formula or a seeded simulation. Change the inputs and see how the conclusions move. Requirements: `numpy`.

In [ ]:
from math import erf, sqrt
import numpy as np

def phi(x):
    """Standard normal CDF."""
    return 0.5 * (1 + erf(x / sqrt(2)))

## 1. How often a Sharpe-0.4 edge loses in a year
Yearly return ~ Normal(mean, vol). Sharpe = mean / vol. P(losing year) = Φ(−Sharpe).

In [ ]:
for sharpe in (0.3, 0.4, 0.5, 3.0):
    p = phi(-sharpe)
    print(f"Sharpe {sharpe}: losing years {p:.2%}  (about 1 year in {1 / p:,.0f})")

## 2. How many years it takes to be reasonably confident
The t-statistic of a mean return grows like Sharpe × √years. Solve t = 2.

In [ ]:
for sharpe in (0.3, 0.4, 0.5):
    print(f"Sharpe {sharpe}: about {(2 / sharpe) ** 2:.0f} years of data for t ≈ 2")

## 3. Breadth
N unrelated markets with equal Sharpe s combine to s·√N. With an average pairwise correlation ρ, s·√(N / (1 + (N − 1)ρ)).

In [ ]:
s, n = 0.15, 10
print(f"unrelated: {s * sqrt(n):.2f}")
for rho in (0.1, 0.3, 0.5):
    print(f"correlation {rho}: {s * sqrt(n / (1 + (n - 1) * rho)):.2f}")

## 4. Slow beats fast (illustrative assumptions)
Cost drag = turnovers per year × cost per full turnover. Net Sharpe = (gross − drag) / volatility.

In [ ]:
gross, vol, cost = 6.0, 10.0, 0.1   # % a year, % a year, % per full turnover
for name, turns in (("monthly", 12), ("weekly", 52), ("daily", 252)):
    drag = turns * cost
    print(f"{name:8s} drag {drag:5.1f}%/yr  net Sharpe {(gross - drag) / vol:+.2f}")

## 5. Every extra try is another chance for a fluke
At a 5% fluke rate per test, P(at least one fluke in k tries) = 1 − 0.95^k (independent tries; real tweaks are correlated, so treat this as the idea, not the exact number).

In [ ]:
for k in (1, 5, 10, 20, 50):
    print(f"{k:3d} tries: {1 - 0.95 ** k:.0%}")

## 6. What a real edge feels like to hold
4,000 simulated 30-year paths at Sharpe 0.4 (4% mean, 10% vol), fixed-size positions (profits not reinvested), seed 3: the same simulation the video's charts use.

In [ ]:
rng = np.random.default_rng(3)
paths, months = 4000, 360
r = rng.normal(0.04 / 12, 0.10 / sqrt(12), (paths, months))
equity = np.cumsum(r, axis=1) * 100                      # % of starting capital
under = equity - np.maximum.accumulate(equity, axis=1)   # distance below the previous high

def longest_run(flags):
    best = run = 0
    for f in flags:
        run = run + 1 if f else 0
        best = max(best, run)
    return best

longest = np.array([longest_run(row < -1e-9) for row in under]) / 12
yearly = r.reshape(paths, 30, 12).sum(axis=2)
three = np.mean([np.any(np.convolve(y < 0, np.ones(3), "valid") == 3) for y in yearly])
print(f"median longest stretch below a previous high: {np.median(longest):.1f} years")
print(f"paths with a 5+ year stretch: {np.mean(longest >= 5):.0%}")
print(f"paths with 3 losing years in a row: {three:.0%}")
print(f"median worst drawdown: {np.median(under.min(axis=1)):.0f} points of starting capital")

### Try this
- Reinvest profits: replace `np.cumsum(r)` with `np.cumprod(1 + r)` and measure drawdowns in % of the peak. The stretches get longer, not shorter.
- Raise the Sharpe to 0.8 and rerun section 6. How much does holding it improve?

*Simulated results on synthetic data. Education only. Not financial advice.*